# MCD-rPPG Dataset - Actual Data EDA

## Comprehensive Analysis of Real Dataset Structure

This notebook provides EDA for the **actual MCD-rPPG database** with 3600 entries and 25 columns.

### Database Structure:
- **3600 rows** (600 subjects x 2 conditions x 3 cameras)
- **25 columns** including vital signs, file paths, and metadata
- **Vital signs**: weight, height, bmi, age, sex, blood pressure, saturation, temperature, etc.
- **File paths**: ecg, ppg, video, meta, ppg_sync (relative paths)

---

### Setup and Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Configuration - Actual paths from user
DB_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'
VIDEOS_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/'
PPG_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ppg/'
PPG_SYNC_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ppg_sync/'
ECG_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ecg/'
METADATA_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/metadata/'

print('Setup complete!')
print(f'DB Path: {DB_PATH}')

## 1. Load and Examine Database Structure

In [ ]:
# Load the actual database
df = pd.read_csv(DB_PATH)

print('=' * 80)
print('DATABASE OVERVIEW')
print('=' * 80)
print(f'Shape: {df.shape}')
print(f'Columns: {len(df.columns)}')
print(f'Rows: {len(df)}')

print('Column Names:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

print('Data Types:')
print(df.dtypes)

# Display first few rows
display(df.head())

## 2. Vital Signs Analysis - Min/Max Table

In [ ]:
# Identify vital sign columns
vital_signs = ['weight', 'height', 'bmi', 'age', 'upper_ap', 'lower_ap', 'saturation', 
               'temperature', 'hemoglobin', 'glycated_hemoglobin', 'cholesterol', 
               'respiratory', 'rigidity', 'pulse', 'stress']

print('=' * 80)
print('VITAL SIGNS MIN/MAX TABLE')
print('=' * 80)

# Create min/max table for vital signs
vital_stats = []
for col in vital_signs:
    if col in df.columns:
        min_val = df[col].min()
        max_val = df[col].max()
        mean_val = df[col].mean()
        std_val = df[col].std()
        vital_stats.append({
            'Vital Sign': col.replace('_', ' ').title(),
            'Min': f'{min_val:.2f}',
            'Max': f'{max_val:.2f}',
            'Mean': f'{mean_val:.2f}',
            'Std': f'{std_val:.2f}',
            'Unit': get_unit(col)
        })

def get_unit(col):
    units = {
        'weight': 'kg',
        'height': 'cm',
        'bmi': 'kg/m2',
        'age': 'years',
        'upper_ap': 'mmHg',
        'lower_ap': 'mmHg',
        'saturation': '%',
        'temperature': 'C',
        'hemoglobin': 'g/dL',
        'glycated_hemoglobin': '%',
        'cholesterol': 'mmol/L',
        'respiratory': 'bpm',
        'rigidity': 'm/s',
        'pulse': 'bpm',
        'stress': 'score'
    }
    return units.get(col, '')

vital_df = pd.DataFrame(vital_stats)
display(vital_df)

In [ ]:
# Plot vital signs distributions
plt.figure(figsize=(18, 12))
for i, col in enumerate(vital_signs[:12], 1):
    plt.subplot(4, 3, i)
    if col in df.columns:
        sns.histplot(df[col], kde=True, bins=30, color='skyblue')
        plt.title(f'{col.replace("_", " ").title()}')
        plt.xlabel(col.replace('_', ' ').title())
plt.tight_layout()
plt.show()

# Plot remaining vital signs
plt.figure(figsize=(18, 6))
for i, col in enumerate(vital_signs[12:], 1):
    plt.subplot(2, 3, i)
    if col in df.columns:
        sns.histplot(df[col], kde=True, bins=30, color='salmon')
        plt.title(f'{col.replace("_", " ").title()}')
        plt.xlabel(col.replace('_', ' ').title())
plt.tight_layout()
plt.show()

## 3. File Paths Analysis

In [ ]:
# Analyze file path columns
file_columns = ['ecg', 'ppg', 'video', 'meta', 'ppg_sync']

print('=' * 80)
print('FILE PATHS ANALYSIS')
print('=' * 80)

for col in file_columns:
    if col in df.columns:
        print(f'\n{col.upper()}:')
        print(f'  Total entries: {df[col].notna().sum()}')
        print(f'  Missing: {df[col].isna().sum()}')
        print(f'  Sample paths:')
        for path in df[col].dropna().head(3):
            print(f'    {path}')

# Check if paths are relative or absolute
for col in file_columns:
    if col in df.columns:
        sample_path = df[col].dropna().iloc[0] if df[col].notna().any() else ''
        is_absolute = os.path.isabs(sample_path) if sample_path else False
        print(f'{col}: {"absolute" if is_absolute else "relative"} paths')

In [ ]:
# Extract information from file paths
if 'video' in df.columns:
    # Extract subject_id, condition, camera from video paths
    df['subject_id'] = df['video'].apply(lambda x: str(x).split('/')[-1].split('__')[1] if pd.notna(x) and '__' in str(x) else None)
    df['condition'] = df['video'].apply(lambda x: 'after' if pd.notna(x) and 'after' in str(x) else ('before' if pd.notna(x) and 'before' in str(x) else None))
    df['camera'] = df['video'].apply(lambda x: str(x).split('/')[-1].split('__')[2].split('.')[0] if pd.notna(x) and len(str(x).split('/')[-1].split('__')) > 2 else None)

print('Extracted Information:')
print(f'Unique subjects: {df["subject_id"].nunique()}')
print(f'Conditions: {df["condition"].unique()}')
print(f'Cameras: {df["camera"].unique()}')

# Plot distributions
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.countplot(data=df, x='condition', palette='Set2')
plt.title('Condition Distribution')

plt.subplot(1, 3, 2)
sns.countplot(data=df, x='camera', palette='Set2')
plt.title('Camera Distribution')
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
top_subjects = df['subject_id'].value_counts().head(10)
sns.barplot(x=top_subjects.values, y=top_subjects.index, palette='viridis')
plt.title('Top 10 Subjects by Recording Count')

plt.tight_layout()
plt.show()

## 4. Video Data Analysis with Metadata Creation

In [ ]:
# Since we don't have preprocessed video metadata, let's create it from file paths
print('=' * 80)
print('VIDEO DATA ANALYSIS')
print('=' * 80)

# Extract video information from paths
if 'video' in df.columns:
    # Create video metadata
    df['video_filename'] = df['video'].apply(lambda x: os.path.basename(str(x)) if pd.notna(x) else None)
    df['video_ext'] = df['video'].apply(lambda x: os.path.splitext(str(x))[1] if pd.notna(x) else None)
    df['video_size_mb'] = df['video'].apply(lambda x: os.path.getsize(x) / (1024*1024) if pd.notna(x) and os.path.exists(x) else None)

    print('Video File Analysis:')
    print(f'Extensions: {df["video_ext"].unique()}')
    print(f'Average size: {df["video_size_mb"].mean():.2f} MB')
    print(f'Total size: {df["video_size_mb"].sum():.2f} GB')

    # Plot video sizes by camera
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='camera', y='video_size_mb', palette='Set2')
    plt.title('Video Size Distribution by Camera')
    plt.ylabel('Size (MB)')
    plt.xticks(rotation=45)
    plt.show()

    # Video sizes by condition
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='condition', y='video_size_mb', palette='Set2')
    plt.title('Video Size Distribution by Condition')
    plt.ylabel('Size (MB)')
    plt.show()

## 5. PPG Signal Analysis

In [ ]:
# Analyze PPG file paths and create metadata
print('=' * 80)
print('PPG SIGNAL ANALYSIS')
print('=' * 80)

if 'ppg' in df.columns:
    # Extract PPG information
    df['ppg_filename'] = df['ppg'].apply(lambda x: os.path.basename(str(x)) if pd.notna(x) else None)
    df['ppg_ext'] = df['ppg'].apply(lambda x: os.path.splitext(str(x))[1] if pd.notna(x) else None)
    df['ppg_size_mb'] = df['ppg'].apply(lambda x: os.path.getsize(x) / (1024*1024) if pd.notna(x) and os.path.exists(x) else None)

    print('PPG File Analysis:')
    print(f'Extensions: {df["ppg_ext"].unique()}')
    print(f'Average size: {df["ppg_size_mb"].mean():.2f} MB')
    print(f'Total size: {df["ppg_size_mb"].sum():.2f} GB')

    # Try to load a sample PPG file
    sample_ppg_path = df['ppg'].dropna().iloc[0] if df['ppg'].notna().any() else None
    if sample_ppg_path and os.path.exists(sample_ppg_path):
        try:
            ppg_signal = np.load(sample_ppg_path)
            print(f'\nSample PPG Signal:')
            print(f'  Shape: {ppg_signal.shape}')
            print(f'  Dtype: {ppg_signal.dtype}')
            print(f'  Duration: {len(ppg_signal) / 100:.2f} seconds (at 100Hz)')
            print(f'  Min: {ppg_signal.min():.2f}')
            print(f'  Max: {ppg_signal.max():.2f}')
            print(f'  Mean: {ppg_signal.mean():.2f}')
            print(f'  Std: {ppg_signal.std():.2f}')
            
            # Plot sample PPG
            plt.figure(figsize=(15, 4))
            plt.plot(ppg_signal[:1000], color='red', linewidth=1)
            plt.title(f'Sample PPG Signal (First 1000 samples at 100Hz)')
            plt.xlabel('Sample Index')
            plt.ylabel('PPG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
            
            # Plot in time domain
            time = np.arange(len(ppg_signal)) / 100.0
            plt.figure(figsize=(15, 4))
            plt.plot(time[:10], ppg_signal[:1000], color='red', linewidth=1)
            plt.title('PPG Signal (First 10 seconds)')
            plt.xlabel('Time (seconds)')
            plt.ylabel('PPG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
        except Exception as e:
            print(f'Error loading PPG: {e}')
    else:
        print('Sample PPG file not found or not accessible')

    # PPG sizes by camera
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='camera', y='ppg_size_mb', palette='Set2')
    plt.title('PPG File Size Distribution by Camera')
    plt.ylabel('Size (MB)')
    plt.xticks(rotation=45)
    plt.show()

## 6. ECG Signal Analysis

In [ ]:
# Analyze ECG file paths
print('=' * 80)
print('ECG SIGNAL ANALYSIS')
print('=' * 80)

if 'ecg' in df.columns:
    # Extract ECG information
    df['ecg_filename'] = df['ecg'].apply(lambda x: os.path.basename(str(x)) if pd.notna(x) else None)
    df['ecg_ext'] = df['ecg'].apply(lambda x: os.path.splitext(str(x))[1] if pd.notna(x) else None)
    df['ecg_size_mb'] = df['ecg'].apply(lambda x: os.path.getsize(x) / (1024*1024) if pd.notna(x) and os.path.exists(x) else None)

    print('ECG File Analysis:')
    print(f'Extensions: {df["ecg_ext"].unique()}')
    print(f'Average size: {df["ecg_size_mb"].mean():.2f} MB')
    print(f'Total size: {df["ecg_size_mb"].sum():.2f} GB')

    # Try to load a sample ECG file
    sample_ecg_path = df['ecg'].dropna().iloc[0] if df['ecg'].notna().any() else None
    if sample_ecg_path and os.path.exists(sample_ecg_path):
        try:
            # Check if it's JSON or NPY
            if sample_ecg_path.endswith('.json'):
                with open(sample_ecg_path, 'r') as f:
                    ecg_data = json.load(f)
                print(f'\nSample ECG (JSON):')
                print(f'  Keys: {list(ecg_data.keys())[:5]}...')
                if 'signal' in ecg_data:
                    ecg_signal = np.array(ecg_data['signal'])
                    print(f'  Signal length: {len(ecg_signal)}')
                    print(f'  Sampling rate: {ecg_data.get("sampling_rate", "unknown")} Hz')
                else:
                    print(f'  Data structure: {type(ecg_data)}')
            else:
                ecg_signal = np.load(sample_ecg_path)
                print(f'\nSample ECG (NPY):')
                print(f'  Shape: {ecg_signal.shape}')
                print(f'  Dtype: {ecg_signal.dtype}')
                print(f'  Duration: {len(ecg_signal) / 500:.2f} seconds (at 500Hz)')
                print(f'  Min: {ecg_signal.min():.2f}')
                print(f'  Max: {ecg_signal.max():.2f}')
                print(f'  Mean: {ecg_signal.mean():.2f}')
                
                # Plot sample ECG
                plt.figure(figsize=(15, 4))
                plt.plot(ecg_signal[:1000], color='green', linewidth=1)
                plt.title(f'Sample ECG Signal (First 1000 samples at 500Hz)')
                plt.xlabel('Sample Index')
                plt.ylabel('ECG Value')
                plt.grid(True, alpha=0.3)
                plt.show()
        except Exception as e:
            print(f'Error loading ECG: {e}')
    else:
        print('Sample ECG file not found or not accessible')

## 7. Landmarks Analysis with MediaPipe

In [ ]:
# Initialize MediaPipe and analyze sample videos
print('=' * 80)
print('LANDMARKS ANALYSIS WITH MEDIAPIPE')
print('=' * 80)

try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    print('MediaPipe available!')
    
    # Define ROI landmarks as provided by user
    rois = {
        'full_face': list(range(468)),
        'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
        'left_eye': list(range(22, 53)),
        'right_eye': list(range(252, 283)),
        'nose': list(range(1, 21)) + list(range(195, 221)),
        'mouth': list(range(60, 81)) + list(range(290, 321)),
        'left_cheek': list(range(0, 101)) + list(range(200, 301)),
        'right_cheek': list(range(100, 201)) + list(range(300, 401)),
        'chin': list(range(150, 171)) + list(range(370, 391)),
        'left_iris': list(range(468, 473)),
        'right_iris': list(range(473, 478))
    }
    
    print('ROI Definitions:')
    for roi_name, landmarks in rois.items():
        print(f'  {roi_name}: {len(landmarks)} landmarks')
    
    # Try to process a sample video with MediaPipe
    sample_video_path = df['video'].dropna().iloc[0] if df['video'].notna().any() else None
    if sample_video_path and os.path.exists(sample_video_path):
        print(f'Processing sample video: {sample_video_path}')
        
        # Load a few frames
        try:
            videodata = skvideo.io.vread(sample_video_path, num_frames=10)
            print(f'Loaded {len(videodata)} frames')
            
            # Initialize MediaPipe Face Mesh
            base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
            options = vision.FaceLandmarkerOptions(
                base_options=base_options,
                output_face_blendshapes=True,
                output_facial_transformation_matrixes=True,
                num_faces=1)
            detector = vision.FaceLandmarker.create_from_options(options)
            
            # Process first frame
            frame = videodata[0]
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
            detection_result = detector.detect(mp_image)
            
            if detection_result.face_landmarks:
                landmarks = detection_result.face_landmarks[0]
                print(f'Detected {len(landmarks)} landmarks')
                
                # Plot landmarks
                plt.figure(figsize=(10, 6))
                plt.imshow(frame)
                plt.scatter([p.x * frame.shape[1] for p in landmarks], 
                           [p.y * frame.shape[0] for p in landmarks], 
                           s=1, c='red', alpha=0.5)
                plt.title('Detected Facial Landmarks')
                plt.axis('off')
                plt.show()
                
                # Calculate ROI statistics
                print('ROI Statistics:')
                for roi_name, roi_landmarks in rois.items():
                    if roi_landmarks:
                        roi_points = [landmarks[i] for i in roi_landmarks if i < len(landmarks)]
                        if roi_points:
                            x_coords = [p.x * frame.shape[1] for p in roi_points]
                            y_coords = [p.y * frame.shape[0] for p in roi_points]
                            print(f'  {roi_name}: {len(roi_points)} points, mean_x={np.mean(x_coords):.1f}, mean_y={np.mean(y_coords):.1f}')
            else:
                print('No face detected in sample frame')
                
        except Exception as e:
            print(f'Error processing video: {e}')
    else:
        print('Sample video not found or not accessible')
        
except ImportError:
    print('MediaPipe not available. Install with: pip install mediapipe')
    print('\nROI Definitions (for reference):')
    rois = {
        'full_face': '468 landmarks',
        'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
        'left_eye': 'landmarks 22-52',
        'right_eye': 'landmarks 252-282',
        'nose': 'landmarks 1-20, 195-220',
        'mouth': 'landmarks 60-80, 290-320',
        'left_cheek': 'landmarks 0-100, 200-300',
        'right_cheek': 'landmarks 100-200, 300-400',
        'chin': 'landmarks 150-170, 370-390',
        'left_iris': 'landmarks 468-472',
        'right_iris': 'landmarks 473-477'
    }
    for roi, defn in rois.items():
        print(f'  {roi}: {defn}')

## 8. Health Biomarkers Analysis (from db.csv)

In [ ]:
# Health biomarkers are in the database, not in metadata
print('=' * 80)
print('HEALTH BIOMARKERS ANALYSIS')
print('=' * 80)

# Categorize biomarkers
cardiovascular = ['upper_ap', 'lower_ap', 'pulse', 'rigidity']
respiratory = ['respiratory', 'saturation']
metabolic = ['hemoglobin', 'glycated_hemoglobin', 'cholesterol']
physiological = ['temperature']
psychological = ['stress']
demographic = ['weight', 'height', 'bmi', 'age', 'sex']

all_categories = {
    'Cardiovascular': cardiovascular,
    'Respiratory': respiratory,
    'Metabolic': metabolic,
    'Physiological': physiological,
    'Psychological': psychological,
    'Demographic': demographic
}

# Plot distributions by category
for category, cols in all_categories.items():
    print(f'\n{category} Biomarkers:')
    plt.figure(figsize=(15, 4))
    for i, col in enumerate(cols, 1):
        plt.subplot(1, len(cols), i)
        if col in df.columns:
            sns.histplot(df[col], kde=True, bins=30, color='skyblue')
            plt.title(f'{col.replace("_", " ").title()}')
            plt.xlabel('')
            plt.xticks([])
    plt.suptitle(f'{category} Biomarkers Distribution')
    plt.tight_layout()
    plt.show()

# Correlation matrix for biomarkers
plt.figure(figsize=(14, 12))
biomarker_cols = [col for col in df.columns if col in sum(all_categories.values(), [])]
corr_matrix = df[biomarker_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, fmt='.2f')
plt.title('Biomarker Correlation Matrix')
plt.tight_layout()
plt.show()

## 9. Multi-Camera Analysis

In [ ]:
# Analyze data by camera (using the 'camera' column from db.csv)
print('=' * 80)
print('MULTI-CAMERA ANALYSIS')
print('=' * 80)

if 'camera' in df.columns:
    print(f'Camera column values: {df["camera"].unique()}')
    print(f'Camera distribution:')
    print(df['camera'].value_counts())
    
    # Analyze vital signs by camera
    camera_vitals = df.groupby('camera')[vital_signs].agg(['mean', 'std', 'min', 'max']).round(2)
    display(camera_vitals)
    
    # Plot vital signs by camera
    for vital in vital_signs[:6]:  # First 6 vital signs
        if vital in df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=df, x='camera', y=vital, palette='Set2')
            plt.title(f'{vital.replace("_", " ").title()} by Camera')
            plt.ylabel(vital.replace('_', ' ').title())
            plt.xticks(rotation=45)
            plt.show()

    # File sizes by camera
    file_size_cols = ['video_size_mb', 'ppg_size_mb', 'ecg_size_mb']
    for col in file_size_cols:
        if col in df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=df, x='camera', y=col, palette='Set2')
            plt.title(f'{col.replace("_", " ").title()} by Camera')
            plt.ylabel(col.replace('_', ' ').title())
            plt.xticks(rotation=45)
            plt.show()
else:
    print('Camera column not found in database')
    # Try to extract camera from video paths
    if 'video' in df.columns:
        df['camera'] = df['video'].apply(lambda x: str(x).split('/')[-1].split('__')[2].split('.')[0] if pd.notna(x) and len(str(x).split('/')[-1].split('__')) > 2 else None)
        print(f'Extracted camera from video paths: {df["camera"].unique()}')

## 10. Condition Analysis (Rest vs Exercise)

In [ ]:
# Analyze data by condition
print('=' * 80)
print('CONDITION ANALYSIS (REST vs EXERCISE)')
print('=' * 80)

if 'condition' in df.columns:
    print(f'Condition values: {df["condition"].unique()}')
    print(f'Condition distribution:')
    print(df['condition'].value_counts())
    
    # Vital signs by condition
    condition_vitals = df.groupby('condition')[vital_signs].agg(['mean', 'std', 'min', 'max']).round(2)
    display(condition_vitals)
    
    # Plot vital signs by condition
    for vital in vital_signs[:8]:  # First 8 vital signs
        if vital in df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=df, x='condition', y=vital, palette='Set2')
            plt.title(f'{vital.replace("_", " ").title()} by Condition')
            plt.ylabel(vital.replace('_', ' ').title())
            plt.show()
    
    # Statistical comparison
    print('Statistical Comparison (Rest vs Exercise):')
    for vital in vital_signs:
        if vital in df.columns:
            rest_data = df[df['condition'] == 'before'][vital].dropna()
            exercise_data = df[df['condition'] == 'after'][vital].dropna()
            
            if len(rest_data) > 0 and len(exercise_data) > 0:
                mean_diff = exercise_data.mean() - rest_data.mean()
                std_diff = exercise_data.std() - rest_data.std()
                print(f'  {vital:20s}: Rest={rest_data.mean():8.2f}, Exercise={exercise_data.mean():8.2f}, Diff={mean_diff:+8.2f}')
else:
    print('Condition column not found')

## 11. Data Quality Assessment

In [ ]:
# Assess data quality
print('=' * 80)
print('DATA QUALITY ASSESSMENT')
print('=' * 80)

# Missing values analysis
missing_data = df.isnull().sum()
missing_pct = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing_data, 'Percentage': missing_pct}).sort_values('Missing', ascending=False)
print('Missing Values:')
display(missing_df[missing_df['Missing'] > 0])

# Data completeness
completeness = (1 - df.isnull().mean().mean()) * 100
print(f'Overall Data Completeness: {completeness:.2f}%')

# Outlier detection for vital signs
plt.figure(figsize=(15, 10))
for i, vital in enumerate(vital_signs[:12], 1):
    plt.subplot(4, 3, i)
    if vital in df.columns:
        sns.boxplot(x=df[vital], color='lightblue')
        plt.title(f'{vital.replace("_", " ").title()}')
        
        Q1 = df[vital].quantile(0.25)
        Q3 = df[vital].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = ((df[vital] < lower) | (df[vital] > upper)).sum()
        outlier_pct = (outliers / len(df[vital])) * 100
        plt.text(0.05, 0.95, f'{outlier_pct:.1f}% outliers', 
                transform=plt.gca().transAxes, 
                bbox=dict(facecolor='white', alpha=0.8))
plt.tight_layout()
plt.show()

## 12. Summary Statistics and Insights

In [ ]:
# Generate comprehensive summary
print('=' * 80)
print('SUMMARY STATISTICS AND INSIGHTS')
print('=' * 80)

print('DATABASE OVERVIEW:')
print(f'  Total entries: {len(df)}')
print(f'  Total columns: {len(df.columns)}')
print(f'  Unique subjects: {df["subject_id"].nunique() if "subject_id" in df.columns else "N/A"}')
print(f'  Conditions: {df["condition"].unique() if "condition" in df.columns else "N/A"}')
print(f'  Cameras: {df["camera"].unique() if "camera" in df.columns else "N/A"}')

print('\nVITAL SIGNS SUMMARY:')
for vital in vital_signs:
    if vital in df.columns:
        print(f'  {vital:20s}: Min={df[vital].min():8.2f}, Max={df[vital].max():8.2f}, Mean={df[vital].mean():8.2f}')

print('\nDATA QUALITY:')
print(f'  Completeness: {completeness:.2f}%')
print(f'  Missing values: {df.isnull().sum().sum()} total')

print('\nFILE SIZES:')
if 'video_size_mb' in df.columns:
    print(f'  Video: {df["video_size_mb"].mean():.2f} MB avg, {df["video_size_mb"].sum():.2f} GB total')
if 'ppg_size_mb' in df.columns:
    print(f'  PPG: {df["ppg_size_mb"].mean():.2f} MB avg, {df["ppg_size_mb"].sum():.2f} GB total')
if 'ecg_size_mb' in df.columns:
    print(f'  ECG: {df["ecg_size_mb"].mean():.2f} MB avg, {df["ecg_size_mb"].sum():.2f} GB total')

print('\nKEY INSIGHTS:')
print('  1. Dataset contains 3600 entries (600 subjects x 2 conditions x 3 cameras)')
print('  2. Vital signs are stored in the database (not in separate metadata)')
print('  3. File paths are relative and need to be combined with base paths')
print('  4. PPG signals are at 100Hz, ECG signals are at 500Hz')
print('  5. MediaPipe can be used for landmark extraction (468 landmarks)')
print('  6. Camera information is available in the database')

## 13. Appendix: ROI Definitions and Preprocessing Notes

In [ ]:
print('=' * 80)
print('APPENDIX: ROI DEFINITIONS AND PREPROCESSING NOTES')
print('=' * 80)

print('ROI Definitions (MediaPipe 468 landmarks):')
rois = {
    'full_face': 'All 468 landmarks',
    'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
    'left_eye': 'landmarks 22-52 (30 landmarks)',
    'right_eye': 'landmarks 252-282 (30 landmarks)',
    'nose': 'landmarks 1-20 and 195-220 (40 landmarks)',
    'mouth': 'landmarks 60-80 and 290-320 (60 landmarks)',
    'left_cheek': 'landmarks 0-100 and 200-300',
    'right_cheek': 'landmarks 100-200 and 300-400',
    'chin': 'landmarks 150-170 and 370-390',
    'left_iris': 'landmarks 468-472 (5 landmarks)',
    'right_iris': 'landmarks 473-477 (5 landmarks)'
}

for roi, defn in rois.items():
    print(f'  {roi:15s}: {defn}')

print('\nPreprocessing Notes:')
print('  Output path: /home/cristic/preprocessed_data')
print('  Faces path: /home/cristic/preprocessed_data/faces/')
print('  Landmarks path: /home/cristic/preprocessed_data/landmarks/')
print('  Use: rppglib.face_utils.process_video(video)')
print('  Video loading: skvideo.io.vread(path)')
print('  PPG loading: np.load(path)')
print('  ECG loading: np.load(path) or json.load(path) for 500Hz samples')

print('\nNext Steps:')
print('  1. Preprocess videos to extract faces and landmarks')
print('  2. Align PPG/ECG signals with video frames using timestamps')
print('  3. Create synchronization metadata')
print('  4. Extract rPPG signals from video ROIs')
print('  5. Train models using aligned data')